Use the torch.nn module to make the deep learning model easily

In [1]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
class Model(nn.Module):
    def __init__(self, num_features):
        # call the constructor function from the nn.Module
        super().__init__()
        
        # the only task of the nn.Lienar is to define a layer of m number of neurons that takes input from n nuerons
        # from previous layers, and perfomrm the linear combination wti the help of the orresponding weight
        self.layer = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()
        self.optimizer = torch.optim.SGD(self.layer.parameters(), lr=0.001)
    
    def forward(self, features):
        out = self.layer(features)
        self.y_pred = self.sigmoid(out).to(torch.float64)
        return self.y_pred
    
    def loss_function(self, real_class):
        loss_fn = nn.BCELoss()
        self.loss = loss_fn(self.y_pred, real_class)
        return self.loss.item()
    
    def optimization(self):
        # it is done so that there won't be previous gradient left which gets added to the new gradient
        self.optimizer.zero_grad()

        # calculate the gradient
        self.loss.backward()

        # backpropagation
        self.optimizer.step()
        return {'weights': self.layer.weight, 'bais' : self.layer.bias}
        

In [3]:
features = torch.rand(10,5)
print(features)
target = torch.randint(low=0, high= 2, size=(10,1), dtype=torch.float64)
print(target)

tensor([[0.6435, 0.4888, 0.6619, 0.7315, 0.9653],
        [0.2826, 0.5511, 0.1713, 0.0511, 0.3914],
        [0.0015, 0.1577, 0.9824, 0.1138, 0.3485],
        [0.7667, 0.0981, 0.1120, 0.7761, 0.3831],
        [0.1758, 0.8481, 0.7110, 0.2103, 0.1061],
        [0.0187, 0.8107, 0.9319, 0.1183, 0.3226],
        [0.4526, 0.0666, 0.7850, 0.1167, 0.5015],
        [0.8928, 0.3179, 0.4972, 0.6686, 0.5783],
        [0.3473, 0.1165, 0.3010, 0.1662, 0.7277],
        [0.2113, 0.7289, 0.7309, 0.4515, 0.9342]])
tensor([[0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.]], dtype=torch.float64)


In [4]:
ann = Model(features.shape[1])


In [5]:
y_pred = ann(features)
y_pred

tensor([[0.5061],
        [0.4673],
        [0.3867],
        [0.5300],
        [0.3978],
        [0.3820],
        [0.4388],
        [0.5091],
        [0.4914],
        [0.4612]], dtype=torch.float64, grad_fn=<ToCopyBackward0>)

In [6]:
ann.layer.weight

Parameter containing:
tensor([[ 0.2209, -0.0546, -0.3536,  0.1388,  0.2637]], requires_grad=True)

In [7]:
ann.layer.bias

Parameter containing:
tensor([-0.2131], requires_grad=True)

In [8]:
loss = 1
for epoch in range(20):
    ann(features)
    loss = ann.loss_function(target)
    print("epoch: ", epoch, "===============>  loss:", loss)
    ann.optimization()

print('weight: ', ann.layer.weight)
print('bias: ', ann.layer.bias)

epoch:  0 ===============>  loss: 0.6989098033888264
epoch:  1 ===============>  loss: 0.6988226366474078
epoch:  2 ===============>  loss: 0.6987355357657419
epoch:  3 ===============>  loss: 0.698648498652581
epoch:  4 ===============>  loss: 0.69856154655232
epoch:  5 ===============>  loss: 0.6984746880975349
epoch:  6 ===============>  loss: 0.6983878860964892
epoch:  7 ===============>  loss: 0.6983011901887851
epoch:  8 ===============>  loss: 0.6982145352319363
epoch:  9 ===============>  loss: 0.69812797895283
epoch:  10 ===============>  loss: 0.6980415358360833
epoch:  11 ===============>  loss: 0.6979551302069232
epoch:  12 ===============>  loss: 0.6978688109981872
epoch:  13 ===============>  loss: 0.6977825866129752
epoch:  14 ===============>  loss: 0.6976964370822404
epoch:  15 ===============>  loss: 0.6976103341391352
epoch:  16 ===============>  loss: 0.6975243562584433
epoch:  17 ===============>  loss: 0.6974384471927708
epoch:  18 ===============>  loss: 0.697352

In [9]:
# ann.layer.parameters() ---> they are the generator so use iteration
for i in ann.layer.parameters():
    print(i)
print("================================================")
for i in ann.layer.named_parameters():
    print(i)

Parameter containing:
tensor([[ 0.2187, -0.0567, -0.3549,  0.1361,  0.2612]], requires_grad=True)
Parameter containing:
tensor([-0.2162], requires_grad=True)
('weight', Parameter containing:
tensor([[ 0.2187, -0.0567, -0.3549,  0.1361,  0.2612]], requires_grad=True))
('bias', Parameter containing:
tensor([-0.2162], requires_grad=True))


In [124]:
# importing the data as the dataframe from csv file
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [125]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [126]:
# rempving the unwanted features like id and unnamed: 32
df.drop(columns=['id', "Unnamed: 32"], inplace = True)

In [127]:
df.shape

(569, 31)

In [128]:
features = df.iloc[:,1:]
target = df.iloc[:,0]

In [129]:
# split the data set into train and test
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=43)

In [130]:
x_train

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
415,11.89,21.17,76.39,433.8,0.09773,0.08120,0.025550,0.021790,0.2019,0.06290,...,13.05,27.21,85.09,522.9,0.1426,0.21870,0.116400,0.08263,0.3075,0.07351
256,19.55,28.77,133.60,1207.0,0.09260,0.20630,0.178400,0.114400,0.1893,0.06232,...,25.05,36.27,178.60,1926.0,0.1281,0.53290,0.425100,0.19410,0.2818,0.10050
420,11.57,19.04,74.20,409.7,0.08546,0.07722,0.054850,0.014280,0.2031,0.06267,...,13.07,26.98,86.43,520.5,0.1249,0.19370,0.256000,0.06664,0.3035,0.08284
448,14.53,19.34,94.25,659.7,0.08388,0.07800,0.088170,0.029250,0.1473,0.05746,...,16.30,28.39,108.10,830.5,0.1089,0.26490,0.377900,0.09594,0.2471,0.07463
195,12.91,16.33,82.53,516.4,0.07941,0.05366,0.038730,0.023770,0.1829,0.05667,...,13.88,22.00,90.81,600.6,0.1097,0.15060,0.176400,0.08235,0.3024,0.06949
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16,14.68,20.13,94.74,684.5,0.09867,0.07200,0.073950,0.052590,0.1586,0.05922,...,19.07,30.88,123.40,1138.0,0.1464,0.18710,0.291400,0.16090,0.3029,0.08216
58,13.05,19.31,82.61,527.2,0.08060,0.03789,0.000692,0.004167,0.1819,0.05501,...,14.23,22.25,90.24,624.1,0.1021,0.06191,0.001845,0.01111,0.2439,0.06289
277,18.81,19.98,120.90,1102.0,0.08923,0.05884,0.080200,0.058430,0.1550,0.04996,...,19.96,24.30,129.00,1236.0,0.1243,0.11600,0.221000,0.12940,0.2567,0.05737
255,13.96,17.05,91.43,602.4,0.10960,0.12790,0.097890,0.052460,0.1908,0.06130,...,16.39,22.07,108.10,826.0,0.1512,0.32620,0.320900,0.13740,0.3068,0.07957


In [131]:
# StandardScaling
scaler = StandardScaler()
# fit_transform will first learn the mean and the s.d. of the trainning data and then do the calculation
# trasnform will that the mean and s.d.of the training data and work for the testing data
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [132]:
x_train_scaled[0]

array([-0.63904656,  0.43769321, -0.64504029, -0.62945156,  0.09920259,
       -0.44019265, -0.79716811, -0.6983249 ,  0.72114937,  0.01190875,
       -0.46133074,  0.00649877, -0.45347087, -0.44690619,  0.96005806,
        0.29047614, -0.61987796, -0.40248677,  0.27735408, -0.64126786,
       -0.67242081,  0.24448954, -0.66651354, -0.63112959,  0.44623997,
       -0.24509593, -0.76226753, -0.49515846,  0.26325243, -0.58507738])

In [133]:
# LabelEncoder
y_train
encoder = LabelEncoder()
y_train_labeled = encoder.fit_transform(y_train)
y_test_labeled = encoder.transform(y_test)

In [134]:
# as we work on pyTorch, tensor is required for computation
features_train = torch.from_numpy(x_train_scaled).to(torch.float32)
features_test = torch.from_numpy(x_test_scaled).to(torch.float32)
target_train = torch.from_numpy(y_train_labeled).to(torch.float32)
target_test = torch.from_numpy(y_test_labeled).to(torch.float32)

In [135]:
features_train.shape

torch.Size([455, 30])

In [136]:
target_train.shape

torch.Size([455])

In [169]:

# making a multi-layer ANN with the overfitting control method dropout and fast convergene method BatchNormalization

class ANN_Model(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        # ANN model:
        self.model = nn.Sequential(
            nn.Linear(num_features,3),
            nn.ReLU(),
            nn.Linear(3,1),
            nn.Sigmoid()
        )
        # optimizer function: 
        self.optim = torch.optim.SGD(self.model.parameters(), lr = 0.1)

    def forward(self, features):
        self.y_pred = self.model(features)
        return self.y_pred
    
    def loss_function(self, y_real):
        loss_fn = nn.BCELoss()
        self.loss = loss_fn(self.y_pred, y_real)
        return self.loss.item()
        
    def optimizer(self):
        self.optim.zero_grad()
        self.loss.backward()
        self.optim.step()
        

In [170]:
model = ANN_Model(features_train.shape[1])

In [171]:
for i in model.model.named_parameters():
    print(i)

('0.weight', Parameter containing:
tensor([[-0.1635, -0.0379,  0.1706, -0.1755,  0.0331, -0.1200, -0.1301,  0.0679,
         -0.0502,  0.0265,  0.0721, -0.0872, -0.0558,  0.0961, -0.0756, -0.1682,
          0.0611,  0.0744, -0.1682, -0.1577,  0.0236,  0.1595, -0.1480,  0.1671,
         -0.1708, -0.1534,  0.0909, -0.0866,  0.0985,  0.1497],
        [-0.0403, -0.0892,  0.1200,  0.0613, -0.0539, -0.0502, -0.1137,  0.0060,
          0.0039,  0.0894, -0.1636, -0.0185, -0.0788, -0.1059, -0.0369,  0.1196,
          0.1068,  0.1582,  0.1729, -0.0449, -0.0364, -0.0480,  0.0821,  0.0003,
         -0.1289, -0.1043, -0.1618, -0.0915, -0.1138,  0.1339],
        [-0.0166,  0.1633,  0.1655,  0.1223, -0.0010,  0.1456, -0.1129, -0.1035,
         -0.1797, -0.1331, -0.1598,  0.0409,  0.0924, -0.0308,  0.0371,  0.1604,
          0.0940,  0.0787, -0.0619,  0.1004,  0.0234,  0.0394,  0.0955, -0.0854,
          0.0898, -0.1536, -0.0189, -0.0644,  0.1110, -0.1370]],
       requires_grad=True))
('0.bias', Para

In [172]:
epoch = 20

In [173]:
target_train_dim = target_train.reshape(features_train.shape[0],1).clone()

In [174]:
y_pred

tensor([8.0367e-06], grad_fn=<SigmoidBackward0>)

## Stochastic Gradient Descent

In [175]:
# Stochastic Gradient Descent
for i in range(1):
    print("epoch:", i)
    for data_record in range(features_train.shape[0]):
        y_pred = model(features_train[data_record])
        print("data_record", data_record, "predicted_value", y_pred)
        loss = model.loss_function(target_train_dim[data_record])
        print("loss", loss)
        model.optimizer()

epoch: 0
data_record 0 predicted_value tensor([0.5890], grad_fn=<SigmoidBackward0>)
loss 0.8891296982765198
data_record 1 predicted_value tensor([0.5450], grad_fn=<SigmoidBackward0>)
loss 0.6069965958595276
data_record 2 predicted_value tensor([0.5631], grad_fn=<SigmoidBackward0>)
loss 0.8281326293945312
data_record 3 predicted_value tensor([0.5167], grad_fn=<SigmoidBackward0>)
loss 0.7270515561103821
data_record 4 predicted_value tensor([0.4746], grad_fn=<SigmoidBackward0>)
loss 0.6435790657997131
data_record 5 predicted_value tensor([0.5581], grad_fn=<SigmoidBackward0>)
loss 0.5831400156021118
data_record 6 predicted_value tensor([0.5690], grad_fn=<SigmoidBackward0>)
loss 0.5638566017150879
data_record 7 predicted_value tensor([0.5390], grad_fn=<SigmoidBackward0>)
loss 0.7742968201637268
data_record 8 predicted_value tensor([0.5286], grad_fn=<SigmoidBackward0>)
loss 0.7520526647567749
data_record 9 predicted_value tensor([0.5533], grad_fn=<SigmoidBackward0>)
loss 0.591791033744812
da

## Batch Gradient Descent

In [192]:
# Batch Gradient Descent
for i in range(epoch):
    y_pred  = model.forward(features_train)
    loss = model.loss_function(target_train_dim)
    model.optimizer()
    print("epoch:",i, "loss:", loss)

epoch: 0 loss: 0.0825735405087471
epoch: 1 loss: 0.08136510848999023
epoch: 2 loss: 0.08014874905347824
epoch: 3 loss: 0.07891526073217392
epoch: 4 loss: 0.07775707542896271
epoch: 5 loss: 0.07666845619678497
epoch: 6 loss: 0.0756441280245781
epoch: 7 loss: 0.07467931509017944
epoch: 8 loss: 0.0737699419260025
epoch: 9 loss: 0.0729183778166771
epoch: 10 loss: 0.07212629169225693
epoch: 11 loss: 0.07137759029865265
epoch: 12 loss: 0.07066915184259415
epoch: 13 loss: 0.0699981302022934
epoch: 14 loss: 0.06936191022396088
epoch: 15 loss: 0.0687527060508728
epoch: 16 loss: 0.06814304739236832
epoch: 17 loss: 0.06756400316953659
epoch: 18 loss: 0.06701350957155228
epoch: 19 loss: 0.06648965179920197


In [193]:
y_pred

tensor([[2.4398e-02],
        [9.9999e-01],
        [1.1321e-02],
        [3.0611e-01],
        [7.2964e-03],
        [9.8794e-01],
        [9.9970e-01],
        [3.0595e-03],
        [4.6540e-02],
        [9.9999e-01],
        [9.9994e-01],
        [1.0584e-03],
        [5.6681e-04],
        [2.3422e-01],
        [4.0042e-04],
        [9.9896e-01],
        [1.1036e-02],
        [6.4903e-02],
        [5.6647e-02],
        [4.5667e-03],
        [6.4396e-04],
        [8.3752e-03],
        [2.4477e-02],
        [2.1479e-05],
        [5.6659e-03],
        [2.1957e-02],
        [8.2498e-03],
        [1.8964e-03],
        [9.9995e-01],
        [3.7584e-03],
        [6.7038e-03],
        [1.2132e-04],
        [4.1687e-01],
        [9.0492e-01],
        [1.0175e-01],
        [4.5831e-04],
        [9.9959e-01],
        [3.1836e-03],
        [1.6334e-04],
        [1.0000e+00],
        [9.5169e-01],
        [5.6588e-03],
        [1.4965e-02],
        [9.9798e-01],
        [3.5738e-01],
        [9

In [194]:
for i in model.model.parameters():
    print(i)

Parameter containing:
tensor([[-0.4034, -0.3353, -0.0566, -0.4176,  0.0713, -0.0378, -0.2689, -0.0986,
         -0.0191,  0.2033, -0.2045, -0.0803, -0.2664, -0.1264,  0.1621,  0.1053,
          0.2676,  0.1856,  0.0482,  0.1469, -0.3048, -0.3033, -0.4532, -0.1297,
         -0.3686, -0.2804, -0.1523, -0.3665, -0.3328,  0.0561],
        [ 0.3270,  0.3618,  0.4872,  0.4085,  0.2670,  0.0275,  0.3407,  0.4684,
         -0.0224, -0.2412,  0.0954, -0.0290,  0.0931,  0.1268,  0.3933, -0.0291,
          0.1735,  0.4075,  0.4411, -0.3190,  0.4155,  0.5655,  0.5161,  0.3993,
          0.6028,  0.0358,  0.2922,  0.4337,  0.4304,  0.0872],
        [-0.4122, -0.5342, -0.2190, -0.2675,  0.0452,  0.2554, -0.4297, -0.4194,
         -0.0935,  0.3035, -0.5271,  0.0521, -0.1427, -0.3617, -0.1572,  0.4780,
          0.0951, -0.0809, -0.0542,  0.4189, -0.4714, -0.8406, -0.3718, -0.5348,
         -0.1998, -0.2925, -0.4292, -0.4917, -0.5416, -0.2059]],
       requires_grad=True)
Parameter containing:
tensor(

In [195]:
# Testing
y = model(features_test)
model.loss_function(target_test.reshape(features_test.shape[0],1))

0.12165968120098114

In [196]:
check = []
for i in y.reshape(features_test.shape[0],):
    if i < 0.5:
        check.append(0)
    else:
        check.append(1)

In [197]:
target_test

tensor([0., 0., 1., 1., 1., 0., 0., 0., 1., 1., 0., 0., 0., 0., 1., 1., 0., 1.,
        0., 1., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0.,
        0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 1.,
        0., 0., 1., 0., 1., 0., 0., 1., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0.,
        1., 1., 1., 0., 0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 1., 1., 0., 1.,
        0., 0., 0., 0., 0., 0.])

In [198]:
z = 0
for i in range(len(target_test)):
    if target_test[i] == check[i]:
        z+=1 
    else:
        print(target_test[i], check[i])
print(z)

tensor(0.) 1
tensor(1.) 0
tensor(0.) 1
tensor(0.) 1
tensor(0.) 1
109


Now we can u se the droput layer and also apply the BatchNormalization in each output of the neuron in each of the hidden layer

Dropout layer is used to dropout or remove/disable certain number of neuron so that the neurons can learn the general pattern through every feature/e=neruon

## Dataset and DataLoaders

In [300]:
from sklearn.datasets import make_classification
import torch
from torch.utils.data import Dataset, DataLoader

In [301]:
x, y = make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42
)

In [302]:
x

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [303]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [304]:
x = torch.from_numpy(x)
y = torch.from_numpy(y).to(torch.float64)

In [305]:
class CustomDataset(Dataset):

    def __init__(self, features, label):
        # featrue and label
        self.features = features
        self.labels = label

    def __len__(self):
        # return the number of the records in the dataset
        return self.features.shape[0]

    def __getitem__(self, index):
        # return the feature and the label of the index passed
        return (self.features[index], self.labels[index])

In [306]:
dataset = CustomDataset(x, y)

In [307]:
dataset.__len__()

10

In [308]:
feature, label = dataset.__getitem__(2)

In [309]:
feature

tensor([-2.8954,  1.9769], dtype=torch.float64)

In [310]:
label

tensor(0., dtype=torch.float64)

### DataLoader

In [316]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [318]:
for batch_feature, batch_label in dataloader:
    print(batch_feature)
    print(batch_label)

tensor([[-2.8954,  1.9769],
        [ 1.7273, -1.1858]], dtype=torch.float64)
tensor([0., 1.], dtype=torch.float64)
tensor([[-0.5872, -1.9717],
        [-0.9382, -0.5430]], dtype=torch.float64)
tensor([0., 1.], dtype=torch.float64)
tensor([[-1.1402, -0.8388],
        [ 1.8997,  0.8344]], dtype=torch.float64)
tensor([0., 1.], dtype=torch.float64)
tensor([[ 1.0683, -0.9701],
        [ 1.7774,  1.5116]], dtype=torch.float64)
tensor([1., 1.], dtype=torch.float64)
tensor([[-0.7206, -0.9606],
        [-1.9629, -0.9923]], dtype=torch.float64)
tensor([0., 0.], dtype=torch.float64)
